# Fine-tuning a masked language model

## Load Dataset:

In [1]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/imdb")

## Load Tokenizer:

In [2]:
from transformers import AutoTokenizer

ckpt = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(ckpt)

## Load Model:

In [3]:
from transformers import AutoModelForMaskedLM
import torch

ckpt = "distilbert-base-uncased"

model = AutoModelForMaskedLM.from_pretrained(
    ckpt, 
    attn_implementation="flash_attention_3", 
    dtype=torch.bfloat16,
    device_map="auto"
)

[transformers] You are attempting to use Flash Attention 3 with dropout. This might lead to unexpected behaviour as this is not supported on recent versions of Flash Attention.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## Tokenize Dataset:

In [4]:
tokenized_datasets = raw_datasets.map(
    function=lambda x: tokenizer(x['text'], truncation=True, max_length=512), 
    batched=True, 
    remove_columns=["text", "label"]
)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

## Data Collator:

In [5]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

## Domain Adapt Model:

In [6]:
from transformers import TrainingArguments
from transformers import Trainer

In [ ]:
args = TrainingArguments(
    output_dir="distilbert-mlm-imdb",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=16,
    bf16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["unsupervised"],
    eval_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [12]:
trainer.train()

Step,Training Loss
500,2.323614
1000,2.331560
1500,2.331961
2000,2.334883
2500,2.339022
3000,2.327095
3500,2.342016
4000,2.333568
4500,2.332815
5000,2.338302


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9375, training_loss=2.33625640625, metrics={'train_runtime': 314.6921, 'train_samples_per_second': 476.656, 'train_steps_per_second': 29.791, 'total_flos': 1.969516606102387e+16, 'train_loss': 2.33625640625, 'epoch': 3.0})

In [ ]:
import math

results = trainer.evaluate()
perplexity = math.exp(results["eval_loss"])
print(f"Eval loss:      {results['eval_loss']:.4f}")
print(f"Perplexity:     {perplexity:.2f}")
results